# City-Day Air Quality Index (AQI) Prediction

This notebook builds and evaluates reproducible regression pipelines for estimating a city-day AQI from same-day pollutant measurements, city, and calendar fields. It expects `city_day.csv` in the notebook's working directory. The chronological holdout evaluates later dates; this is an AQI estimation task, not a forecast made before same-day pollutant measurements are available.

## 1. Project Overview and Problem Definition

The target is `AQI`, a numeric field. `AQI_Bucket` is excluded because it is a category associated with the AQI and could disclose target information. The inputs are city, date-derived calendar features, and pollutant columns. We measure generalization to dates later than the training period, select models using time-series cross-validation on training data, and report the untouched final period once. MAE is the primary selection metric because it expresses typical error in AQI points and is less sensitive to a few extreme errors than squared error. R² and RMSE are supporting metrics.

The dataset provides no accompanying data dictionary, measurement units, AQI calculation formula, or collection methodology. Descriptions below are limited to what field names and observed values establish. Missing target rows cannot train or score a supervised model. City/date coverage and pollutant availability may vary over time, so future deployment requires monitoring for drift and data-quality changes.

## 2. Imports and Environment Setup

Install the required packages if needed: `pip install pandas numpy matplotlib seaborn scikit-learn joblib`. No optional boosting package is required. A fixed seed is used where algorithms are stochastic.

In [ ]:
from pathlib import Path
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import joblib

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit, cross_val_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

SEED = 42
sns.set_theme(style='whitegrid', context='notebook')
print('pandas:', pd.__version__, '| scikit-learn:', sklearn.__version__)

## 3. Dataset Loading

Load the supplied file and parse the date column. Invalid date values are reported; they are excluded from modeling because calendar features and chronological splitting require valid dates.

In [ ]:
DATA_PATH = Path('city_day.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Place city_day.csv in the notebook working directory.')
df = pd.read_csv(DATA_PATH)
if 'Date' not in df or 'AQI' not in df:
    raise ValueError('Expected Date and AQI columns were not found.')
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('Dtypes:\n', df.dtypes)
display(df.head())
display(df.describe(include='all').T)

## 4. Dataset Metadata

The source file contains names but no authoritative definitions or units. The descriptions are deliberately conservative; confirm operational meaning with the data provider before deployment.

In [ ]:
known_notes = {
    'City': 'City label in the source file; geographic coordinate or administrative definition not supplied.',
    'Date': 'Calendar date represented by the row.',
    'AQI': 'Numeric AQI target as named by the source; scale and computation method not supplied.',
    'AQI_Bucket': 'Categorical AQI bucket as named by the source; thresholds not supplied.'
}
for col in df.columns:
    if col not in known_notes:
        known_notes[col] = ('Pollutant measurement field inferred from its label; units and measurement method are not supplied.'
                            if col not in ['Date', 'City'] else '')
metadata_rows = []
for col in df.columns:
    s = df[col]
    examples = s.dropna().astype(str).drop_duplicates().head(4).tolist()
    metadata_rows.append({
        'Feature': col, 'Data type': str(s.dtype), 'Description': known_notes[col],
        'Missing': int(s.isna().sum()), 'Unique': int(s.nunique(dropna=True)),
        'Examples': ', '.join(examples),
        'Summary': (str(s.describe().to_dict()) if pd.api.types.is_numeric_dtype(s) else '')
    })
metadata = pd.DataFrame(metadata_rows)
display(metadata)

## 5. Data Quality Assessment

Review missing values, repeated rows, target coverage, and categorical values before defining the modeling sample.

In [ ]:
print('Duplicate rows:', int(df.duplicated().sum()))
print('Invalid dates:', int(df['Date'].isna().sum()))
print('Rows with missing AQI:', int(df['AQI'].isna().sum()))
display(df.isna().sum().sort_values(ascending=False).rename('missing').to_frame())
for col in ['City', 'AQI_Bucket']:
    if col in df:
        print(f'\n{col} values (top 20):')
        display(df[col].value_counts(dropna=False).head(20).to_frame('count'))

## 6. Exploratory Data Analysis

The charts below summarize observed distributions and relationships. Interpretations are printed from computed values instead of being prefilled with unverified conclusions.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
missing_rates = df.isna().mean().sort_values(ascending=False)

ax = missing_rates.plot.bar(figsize=(10, 4), color="steelblue")
ax.set_title("Missing values by feature")
ax.set_ylabel("Fraction missing")
ax.set_xlabel("Feature")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print("Higher bars indicate features that need more imputation.")

In [ ]:
ax = sns.histplot(data=df, x="AQI", bins=40, kde=True)
ax.set_title("Distribution of AQI")
ax.set_xlabel("AQI")
ax.set_ylabel("City-day count")
plt.tight_layout()
plt.show()

print("AQI summary:")
display(df["AQI"].describe().round(2))

In [ ]:
plot_numeric = [column for column in numeric_cols if column != "AQI"]

if plot_numeric:
    df[plot_numeric].hist(
        figsize=(15, 12), bins=30, grid=False, color="slateblue"
    )
    plt.suptitle("Distributions of numeric measurements")
    plt.tight_layout()
    plt.show()

print("Use these plots to inspect skew and unusual ranges.")

In [ ]:
if plot_numeric:
    fig, ax = plt.subplots(figsize=(14, 5))
    sns.boxplot(data=df[plot_numeric], orient="h", showfliers=False, ax=ax)
    ax.set_title("Numeric measurement ranges")
    ax.set_xlabel("Recorded value")
    plt.tight_layout()
    plt.show()

print("Extreme values are kept unless the source confirms they are errors.")

In [ ]:
corr_cols = [
    column for column in numeric_cols
    if df[column].notna().sum() > 1
]
correlations = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(correlations, cmap="vlag", center=0, ax=ax)
ax.set_title("Correlation among numeric fields")
plt.tight_layout()
plt.show()

print("Correlations show associations, not causes.")

In [ ]:
if "City" in df:
    city_counts = df["City"].value_counts().head(20).sort_values()
    ax = city_counts.plot.barh(figsize=(11, 5), color="teal")
    ax.set_title("Records by city (top 20)")
    ax.set_xlabel("Rows")
    ax.set_ylabel("City")
    plt.tight_layout()
    plt.show()
    print("Cities represented:", df["City"].nunique())

if "AQI_Bucket" in df:
    bucket_counts = df["AQI_Bucket"].value_counts(dropna=False)
    ax = bucket_counts.plot.bar(figsize=(9, 4), color="darkorange")
    ax.set_title("AQI bucket counts (missing included)")
    ax.set_xlabel("AQI bucket")
    ax.set_ylabel("Rows")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()
    print("AQI_Bucket is excluded from the model inputs to avoid leakage.")

In [ ]:
valid_target = df.dropna(subset=["AQI", "Date"]).copy()
valid_target["Year"] = valid_target["Date"].dt.year
year_aqi = valid_target.groupby("Year")["AQI"].agg(
    count="count", mean="mean", median="median"
)
display(year_aqi)

fig, ax = plt.subplots(figsize=(11, 4))
sns.lineplot(data=valid_target, x="Date", y="AQI", errorbar=None, ax=ax)
ax.set_title("AQI over time (cities pooled)")
ax.set_xlabel("Date")
ax.set_ylabel("AQI")
plt.tight_layout()
plt.show()

print("The later-date holdout helps evaluate performance over time.")

## 7. Data Preprocessing and Feature Engineering

Extract calendar components from `Date` before splitting because these are deterministic properties of each record. The fitted imputers, scaling, and city encoder remain inside each pipeline and are learned from training folds only. Exact duplicate records are dropped. No outlier clipping is applied without source-specific validation. `AQI_Bucket` is never an input.

In [ ]:
data = df.drop_duplicates().copy()
data = data.dropna(subset=['AQI', 'Date']).sort_values(['Date', 'City']).reset_index(drop=True)
data['Year'] = data['Date'].dt.year
data['Month'] = data['Date'].dt.month
data['Day'] = data['Date'].dt.day
data['DayOfWeek'] = data['Date'].dt.dayofweek
data['DayOfYear'] = data['Date'].dt.dayofyear
data['IsWeekend'] = (data['DayOfWeek'] >= 5).astype(int)
target = data['AQI'].astype(float)
feature_cols = [c for c in data.columns if c not in ['AQI', 'AQI_Bucket', 'Date']]
X = data[feature_cols].copy()
categorical_features = [c for c in X.columns if X[c].dtype == 'object' or isinstance(X[c].dtype, pd.CategoricalDtype)]
numeric_features = [c for c in X.columns if c not in categorical_features]
print('Modeling rows:', len(X), '| numeric:', numeric_features, '| categorical:', categorical_features)

## 8. Chronological Train-Test Split

Rows are ordered by date. All records on the latest 20% of distinct dates form the held-out test period. This evaluates future-date performance while retaining all cities and records that share a date on the same side of the split.

In [ ]:
unique_dates = np.sort(data['Date'].unique())
cutoff_idx = max(1, int(np.floor(len(unique_dates) * 0.8)))
cutoff_date = pd.Timestamp(unique_dates[cutoff_idx]) if cutoff_idx < len(unique_dates) else pd.Timestamp(unique_dates[-1])
train_mask = data['Date'] < cutoff_date
X_train, X_test = X.loc[train_mask].copy(), X.loc[~train_mask].copy()
y_train, y_test = target.loc[train_mask].copy(), target.loc[~train_mask].copy()
print('Cutoff:', cutoff_date, '| train:', X_train.shape, '| test:', X_test.shape)
print('Train dates:', data.loc[train_mask, 'Date'].min(), 'to', data.loc[train_mask, 'Date'].max())
print('Test dates:', data.loc[~train_mask, 'Date'].min(), 'to', data.loc[~train_mask, 'Date'].max())
if len(X_train) == 0 or len(X_test) == 0:
    raise ValueError('Chronological split produced an empty partition.')

## 9. Baseline Model

The baseline predicts the training-set median AQI for every record. It provides a reference for judging whether feature-based models add value.

In [ ]:
baseline_prediction = np.full(len(y_test), y_train.median())
baseline_metrics = {
    "Model": "Training median baseline",
    "MAE": mean_absolute_error(y_test, baseline_prediction),
    "RMSE": mean_squared_error(y_test, baseline_prediction) ** 0.5,
    "R2": r2_score(y_test, baseline_prediction),
}
display(pd.DataFrame([baseline_metrics]))

## 10. Model Implementation and Training

Each estimator receives the same chronological split and fold-local preprocessing. Six distinct regressors are included; Random Forest and Extra Trees are ensembles. Scaling is included for comparable pipelines and benefits distance/linear models; tree estimators do not depend on it. A dense one-hot representation supports HistGradientBoosting on this modest number of cities.

In [ ]:
numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_preprocessor, numeric_features),
    ("categorical", categorical_preprocessor, categorical_features),
])

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(
        max_depth=18, min_samples_leaf=5, random_state=SEED
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=160, min_samples_leaf=2, n_jobs=1, random_state=SEED
    ),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=160, min_samples_leaf=2, n_jobs=1, random_state=SEED
    ),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(
        max_iter=140, learning_rate=0.08, max_leaf_nodes=20,
        l2_regularization=1.0, random_state=SEED
    ),
    "K-Nearest Neighbors": KNeighborsRegressor(
        n_neighbors=12, weights="distance", n_jobs=1
    ),
}

fitted_models = {}
model_records = []
for name, estimator in models.items():
    pipeline = Pipeline([
        ("preprocess", clone(preprocessor)),
        ("model", clone(estimator)),
    ])

    start_time = time.perf_counter()
    pipeline.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - start_time

    start_time = time.perf_counter()
    predictions = pipeline.predict(X_test)
    predict_seconds = time.perf_counter() - start_time

    fitted_models[name] = pipeline
    model_records.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": mean_squared_error(y_test, predictions) ** 0.5,
        "R2": r2_score(y_test, predictions),
        "Fit seconds": fit_seconds,
        "Predict seconds": predict_seconds,
    })

results = pd.DataFrame(model_records).sort_values("MAE")
display(results)

## 11. Cross-Validation and Model Comparison

TimeSeriesSplit preserves temporal order within the training period. The chronological folds estimate how model errors vary over later training periods. The final test set is not used for selection; it is shown above only as the required initial comparison and will be evaluated after tuning below.

In [ ]:
n_splits = min(4, max(2, len(X_train) // 1000))
ts_cv = TimeSeriesSplit(n_splits=n_splits)
cv_records = []

for name, estimator in models.items():
    pipeline = Pipeline([
        ("preprocess", clone(preprocessor)),
        ("model", clone(estimator)),
    ])
    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=ts_cv,
        scoring="neg_mean_absolute_error",
        n_jobs=1,
    )
    fold_mae = -scores
    cv_records.append({
        "Model": name,
        "CV MAE mean": fold_mae.mean(),
        "CV MAE std": fold_mae.std(),
    })

cv_results = pd.DataFrame(cv_records).sort_values("CV MAE mean")
results = results.merge(cv_results, on="Model")
results = results.sort_values("CV MAE mean")
display(results)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=results, x="CV MAE mean", y="Model", color="cornflowerblue", ax=ax)
ax.set_title("Time-series cross-validation (lower MAE is better)")
ax.set_xlabel("Mean validation MAE (AQI points)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=results, x="Fit seconds", y="Model", color="seagreen", ax=ax)
ax.set_title("Training time")
ax.set_xlabel("Seconds")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

print("Choose based on validation MAE; fit time is a practical tradeoff.")

## 12. Hyperparameter Tuning

Tune the promising HistGradientBoosting model with a small randomized search and chronological folds. Search decisions use only the training partition. The selected estimator is refit by the search on all training rows.

In [ ]:
tuning_pipeline = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", HistGradientBoostingRegressor(random_state=SEED)),
])
parameter_options = {
    "model__learning_rate": [0.04, 0.08, 0.12],
    "model__max_iter": [100, 180, 260],
    "model__max_leaf_nodes": [15, 25, 35],
    "model__l2_regularization": [0.0, 1.0, 5.0],
}
search = RandomizedSearchCV(
    tuning_pipeline,
    param_distributions=parameter_options,
    n_iter=10,
    scoring="neg_mean_absolute_error",
    cv=ts_cv,
    random_state=SEED,
    n_jobs=1,
    refit=True,
)
search.fit(X_train, y_train)

print("Best cross-validation MAE:", -search.best_score_)
print("Best parameters:", search.best_params_)

## 13. Final Model Selection and Untouched Test Evaluation

Select the tuned candidate if its training-only cross-validation MAE beats the best untuned model's cross-validation MAE; otherwise use the best untuned model. Only now score the selected pipeline on the held-out later-date period.

In [ ]:
best_untuned = cv_results.iloc[0]
best_untuned_name = best_untuned["Model"]
best_untuned_mae = best_untuned["CV MAE mean"]
tuned_mae = -search.best_score_

if tuned_mae < best_untuned_mae:
    selected_name = "Tuned Hist Gradient Boosting"
    selected_pipeline = search.best_estimator_
else:
    selected_name = best_untuned_name
    selected_pipeline = fitted_models[best_untuned_name]

selected_prediction = selected_pipeline.predict(X_test)
final_metrics = {
    "Selected model": selected_name,
    "Selection CV MAE": min(tuned_mae, best_untuned_mae),
    "Test MAE": mean_absolute_error(y_test, selected_prediction),
    "Test RMSE": mean_squared_error(y_test, selected_prediction) ** 0.5,
    "Test R2": r2_score(y_test, selected_prediction),
}
display(pd.DataFrame([final_metrics]))
print("Untuned CV leader:", best_untuned_name)
print("Untuned CV MAE:", best_untuned_mae)
print("Tuned CV MAE:", tuned_mae)

## 14. Explainability and Diagnostics

Permutation importance measures the increase in held-out MAE after shuffling each feature. It is a model-agnostic diagnostic, but correlated inputs can share importance and later-period importance may not represent all cities or future regimes. Actual-versus-predicted and residual plots expose systematic errors.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
sns.scatterplot(x=y_test, y=selected_prediction, alpha=0.35, ax=ax)
plot_min = min(y_test.min(), selected_prediction.min())
plot_max = max(y_test.max(), selected_prediction.max())
ax.plot([plot_min, plot_max], [plot_min, plot_max], "--", color="red")
ax.set_title(f"Actual vs predicted - {selected_name}")
ax.set_xlabel("Actual AQI")
ax.set_ylabel("Predicted AQI")
plt.tight_layout()
plt.show()

residuals = y_test.to_numpy() - selected_prediction
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(residuals, kde=True, ax=axes[0])
axes[0].set_title("Residual distribution")
axes[0].set_xlabel("Actual minus predicted AQI")

sns.scatterplot(x=selected_prediction, y=residuals, alpha=0.35, ax=axes[1])
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title("Residuals vs predicted")
axes[1].set_xlabel("Predicted AQI")
axes[1].set_ylabel("Residual")
plt.tight_layout()
plt.show()

In [ ]:
importance = permutation_importance(
    selected_pipeline,
    X_test,
    y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=SEED,
    n_jobs=1,
)
importance_table = pd.DataFrame({
    "Feature": feature_cols,
    "MAE increase": importance.importances_mean,
    "Std": importance.importances_std,
}).sort_values("MAE increase", ascending=False)
display(importance_table.head(15))

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=importance_table.head(15),
    x="MAE increase",
    y="Feature",
    color="mediumpurple",
    ax=ax,
)
ax.set_title("Permutation importance on held-out dates")
ax.set_xlabel("Increase in MAE when shuffled")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 15. Model Serialization

Save the fitted preprocessing-and-model pipeline as one joblib artifact, along with a compact JSON metadata file describing target and expected input schema. The artifact is written to the current working directory.

In [ ]:
artifact_path = Path("aqi_prediction_pipeline.joblib")
metadata_path = Path("aqi_prediction_metadata.json")

joblib.dump(selected_pipeline, artifact_path)
metadata = {
    "target": "AQI",
    "excluded_leakage_field": "AQI_Bucket",
    "input_features": feature_cols,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "date_features": ["Year", "Month", "Day", "DayOfWeek", "DayOfYear", "IsWeekend"],
    "training_cutoff_exclusive": str(cutoff_date.date()),
    "selected_model": selected_name,
    "selection_metric": "TimeSeriesSplit cross-validation MAE",
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

loaded_pipeline = joblib.load(artifact_path)
print("Saved model:", artifact_path.resolve())
print("Saved metadata:", metadata_path.resolve())
print("Sample predictions:", loaded_pipeline.predict(X_test.head(3)))

## 16. Prediction Demonstration

This helper validates required columns, reports unexpected columns, checks that required numeric fields can be interpreted numerically, and aligns the input schema. Missing values are passed to the saved imputers; preprocessing is not refitted during inference.

In [ ]:
def predict_aqi(new_records, pipeline=loaded_pipeline):
    """Predict AQI for rows with the model's input columns."""
    if not isinstance(new_records, pd.DataFrame):
        raise TypeError("new_records must be a pandas DataFrame")

    missing_columns = set(feature_cols) - set(new_records.columns)
    extra_columns = set(new_records.columns) - set(feature_cols)
    if missing_columns:
        raise ValueError(f"Missing input columns: {sorted(missing_columns)}")
    if extra_columns:
        raise ValueError(f"Unexpected input columns: {sorted(extra_columns)}")

    model_input = new_records[feature_cols].copy()
    for column in numeric_features:
        model_input[column] = pd.to_numeric(model_input[column], errors="coerce")

    return pipeline.predict(model_input)


demo = X_test.head(5).copy()
prediction_demo = pd.DataFrame({
    "Predicted AQI": predict_aqi(demo),
    "Actual AQI": y_test.loc[demo.index].to_numpy(),
}, index=demo.index)
display(prediction_demo)

## 17. Technical Conclusions and Limitations

Use the computed comparison tables, validation scores, final later-date metrics, residual plots, and permutation importance above to describe model performance. Do not interpret these estimates as pre-event forecasts: same-day pollutant readings are required inputs. The dataset does not establish measurement units, monitoring-station representativeness, AQI formula, sensor calibration, or missingness mechanism. A temporal holdout is more realistic than a random split for later dates, but records from known cities remain in both partitions; it does not measure generalization to unseen cities. AQI construction may be mathematically linked to pollutant readings, and the exact formula is unavailable. Before deployment, validate source definitions, monitor missingness and drift, evaluate city-specific error, and confirm that input availability matches the intended prediction time.